In [1]:
%pip install matplotlib xgboost shap seaborn scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install openpyxl
import openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Data Manipulation and Analysis
import pandas as pd
import numpy as np

# Visualization (for Pearson correlation matrix and plots)
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Data Split & Tuning
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV

# Machine Learning Models (Regression)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression  # This is for Multivariate Linear Regression
import xgboost as xgb
from xgboost import XGBRegressor

# Model Evaluation Metrics (Regression)
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score  # This represents your R-squared score
)

# Feature Importance
import shap



# Optional: Ignore warnings for cleaner outputs
import warnings
warnings.filterwarnings('ignore')
import os
import glob


c:\Users\Phuc\Downloads\Thesis v2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# 1. Load the NPS specific file
filepath = r'..\Dataset\nps_raw.xlsx'
print(f"Loading {filepath}...")
nps_df = pd.read_excel(filepath)

# Select only the exact columns we need so we don't carry extra weight
cols_to_keep = ['Ngày tạo/ khảo sát', 'Phân loại kết quả', 'Cửa hàng', 'Phương thức']
nps_df = nps_df[cols_to_keep].dropna(how='all').copy()

# 2. Transform the Date into 'YearMonth' level and rename Store
nps_df['Ngày tạo/ khảo sát'] = pd.to_datetime(nps_df['Ngày tạo/ khảo sát'], errors='coerce')
nps_df['YearMonth'] = nps_df['Ngày tạo/ khảo sát'].dt.to_period('M').astype(str)

# Rename to standard format so the master merge script matches it perfectly
nps_df.rename(columns={'Cửa hàng': 'store_id'}, inplace=True)

# Clean string columns to lowercase for safe matching
nps_df['Phân loại kết quả'] = nps_df['Phân loại kết quả'].astype(str).str.lower().str.strip()
nps_df['Phương thức'] = nps_df['Phương thức'].astype(str).str.lower().str.strip()

# 3. First Logic: Categorize based on feedback result
def categorize_satisfaction(val):
    if val.startswith('hài lòng'):
        return 'satisfied_ticket'
    elif val.startswith('bình thường'):
        return 'neutral_ticket'
    else:
        # This catches 'vi phạm', 'khong hai long', and any other strange values
        return 'dissatisfied_ticket'

# Create the ticket distribution features
nps_df['NPS_Cat'] = nps_df['Phân loại kết quả'].apply(categorize_satisfaction)
nps_dummies = pd.get_dummies(nps_df['NPS_Cat'], dtype=int)
nps_df = pd.concat([nps_df, nps_dummies], axis=1)

# Ensure all 3 columns exist even if the dataset happens to be missing one entirely
for col in ['satisfied_ticket', 'neutral_ticket', 'dissatisfied_ticket']:
    if col not in nps_df.columns:
        nps_df[col] = 0

# 4. Second Logic: Categorize based on Method (Phương thức)
def categorize_order(val):
    # Accounting for potential tiny typos in the raw data ('đon' vs 'đơn')
    if val in ['đơn giao hàng', 'Đơn giao hàng']:
        return 'order_delivered_tickets'
    elif val in ['Đơn đến cửa hàng']:
        return 'order_pickup_tickets'
    elif val in ['Bán lẻ']:
        return 'order_onsite_tickets'
    else:
        return 'order_other_tickets'

nps_df['Order_Cat'] = nps_df['Phương thức'].apply(categorize_order)
order_dummies = pd.get_dummies(nps_df['Order_Cat'], dtype=int)
nps_df = pd.concat([nps_df, order_dummies], axis=1)

for col in ['order_delivered_tickets', 'order_pickup_tickets', 'order_onsite_tickets', 'order_other_tickets']:
    if col not in nps_df.columns:
        nps_df[col] = 0

# 5. Aggregate by Region (store_id) AND Month
agg_funcs = {
    'satisfied_ticket': 'sum',
    'neutral_ticket': 'sum',
    'dissatisfied_ticket': 'sum',
    'order_delivered_tickets': 'sum',
    'order_pickup_tickets': 'sum',
    'order_onsite_tickets': 'sum',
    'order_other_tickets': 'sum'
}

# Apply the aggregation
nps_agg = nps_df.groupby(['store_id', 'YearMonth']).agg(agg_funcs).reset_index()

# 6. Calculate Weighted Average Score
total_tickets = nps_agg['satisfied_ticket'] + nps_agg['neutral_ticket'] + nps_agg['dissatisfied_ticket']

# Avoid division by zero by using np.where (assigns NaN if there are 0 tickets)
nps_agg['nps_weighted_score'] = np.where(
    total_tickets > 0,
    (nps_agg['satisfied_ticket'] * 1 + nps_agg['neutral_ticket'] * 3 + nps_agg['dissatisfied_ticket'] * 5) / total_tickets,
    np.nan
)

# 7. Save to the features folder
out_path = r'..\Dataset\features\nps_aggregated_features.csv'
nps_agg.to_csv(out_path, index=False)
print(f"✅ NPS Feature set created securely in: '{out_path}'")

display(nps_agg.head())


Loading ..\Dataset\nps_raw.xlsx...
✅ NPS Feature set created securely in: '..\Dataset\features\nps_aggregated_features.csv'


,store_id,YearMonth,satisfied_ticket,neutral_ticket,dissatisfied_ticket,order_delivered_tickets,order_pickup_tickets,order_onsite_tickets,order_other_tickets,nps_weighted_score
0,CPS-AGI-CDO-272LL,2025-10,1,0,0,0,0,0,1,1.000000
1,CPS-AGI-CDO-272LL,2026-01,124,2,0,1,0,0,125,1.031746
2,CPS-AGI-CDO-272LL,2026-03,0,0,1,0,0,0,1,5.000000
3,CPS-AGI-LXG-1393THD,2025-10,121,9,1,1,0,0,130,1.167939
4,CPS-AGI-LXG-1393THD,2026-01,109,0,0,1,0,0,108,1.000000


In [ ]:
# ── Process shop_info directly from source ──
shop_raw = pd.read_excel(r'..\Dataset\shop_info.xlsx')

cols_to_keep = [
    'ma_shop', 'sizeshop', 'mien', 'tong_dien_tichm2',
    'rong_ngang_m', 'dai_sau_m', 'kv_ban_hangm2',
    'kho_m2', 'via_he_rongm', 'khai_truong_date'
]
shop = shop_raw[cols_to_keep].copy()

# Robust cleaning for numeric columns (handling commas, special strings like '11,2 và 15,5')
def clean_numeric(val):
    if pd.isna(val): return np.nan
    if isinstance(val, (int, float)): return float(val)
    # Replace comma with dot for decimal
    s = str(val).replace(',', '.')
    # Extract the first numeric sequence (including decimals)
    import re
    match = re.search(r"[-+]?\d*\.\d+|\d+", s)
    return float(match.group()) if match else np.nan

num_cols = ['tong_dien_tichm2', 'rong_ngang_m', 'dai_sau_m', 'kv_ban_hangm2', 'kho_m2', 'via_he_rongm']
for col in num_cols:
    shop[col] = shop[col].apply(clean_numeric)

# Map sizeshop: A=4, B=3, C=2, D=1
size_mapping = {'A': 4, 'B': 3, 'C': 2, 'D': 1}
shop['sizeshop'] = shop['sizeshop'].astype(str).str.upper().str.strip().map(size_mapping)

# is_southern: NAM=1, else 0
shop['mien'] = shop['mien'].astype(str).str.upper().str.strip()
shop['is_southern'] = (shop['mien'] == 'NAM').astype(int)
shop.drop(columns=['mien'], inplace=True)

# Convert opening date to datetime
shop['khai_truong_date'] = pd.to_datetime(shop['khai_truong_date'], errors='coerce')

# Rename to match merge key
shop.rename(columns={'ma_shop': 'store_id'}, inplace=True)

# 7. Save to the features folder
out_path = r'..\Dataset\features\shop.csv'
shop.to_csv(out_path, index=False)
print(f"✅ NPS Feature set created securely in: '{out_path}'")



✅ NPS Feature set created securely in: '..\Dataset\features\shop.csv'


In [ ]:
# %%
# pip install matplotlib xgboost shap seaborn scikit-learn
# Data Manipulation and Analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Data Split & Tuning
from sklearn.model_selection import train_test_split, GridSearchCV

# Machine Learning Models (Regression)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Model Evaluation Metrics (Regression)
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# Feature Importance
import shap

# Optional: Ignore warnings for cleaner outputs
import warnings
warnings.filterwarnings('ignore')
import os


# %%
# ═══════════════════════════════════════════════════════════
# SECTION 1 — LOAD ALL FEATURE TABLES
# ═══════════════════════════════════════════════════════════
print("Loading feature tables...")

base_dir = r'c:\Users\Phuc\Downloads\Thesis v2\Dataset\features'

nps   = pd.read_csv(os.path.join(base_dir, 'nps_aggregated_features.csv'))
txn   = pd.read_csv(os.path.join(base_dir, 'transaction_features.csv'))
deliv = pd.read_excel(os.path.join(base_dir, 'delivery_features.xlsx'))
plano = pd.read_csv(os.path.join(base_dir, 'planogram_features.csv'))
shop  = pd.read_csv(os.path.join(base_dir, 'shop.csv')) 
ms = pd.read_excel(os.path.join(base_dir, 'ms_features.xslx'))

# Convert opening date to datetime if not already
shop['khai_truong_date'] = pd.to_datetime(shop['khai_truong_date'], errors='coerce')


print(f"  NPS: {nps.shape} | TXN: {txn.shape} | DELIV: {deliv.shape} | PLANO: {plano.shape} | SHOP: {shop.shape}")

# ═══════════════════════════════════════════════════════════
# SECTION 2 — MERGE & TENURE
# ═══════════════════════════════════════════════════════════
# Merge datasets starting from a base combination (txn and nps)
merged = pd.merge(txn, nps, on=['store_id', 'YearMonth'], how='inner')
merged = pd.merge(merged, deliv, on=['store_id', 'YearMonth'], how='left')  
merged = pd.merge(merged, plano, on=['store_id', 'YearMonth'], how='left')

# Filter to Oct 2025 – Feb 2026
merged = merged[merged['YearMonth'].between('2025-10', '2026-02')].copy()

# Merge with shop_info
merged = pd.merge(merged, shop, on='store_id', how='left')

# ── Create tenure column ──
merged['_period'] = pd.to_datetime(merged['YearMonth'], format='%Y-%m')
merged['tenure'] = (
    (merged['_period'].dt.year  - merged['khai_truong_date'].dt.year)  * 12 +
    (merged['_period'].dt.month - merged['khai_truong_date'].dt.month)
)
merged.drop(columns=['_period', 'khai_truong_date'], inplace=True)

print(f"\nFinal merged dataset: {merged.shape}")
print(f"Months: {sorted(merged['YearMonth'].unique())}")


# ═══════════════════════════════════════════════════════════
# SECTION 3 — FEATURE / TARGET SPLIT & PEARSON CORRELATION
# ═══════════════════════════════════════════════════════════
TARGET    = 'nps_weighted_score'

# Exclude identification columns and calculation intermediates from features
DROP_COLS = [
    'store_id', 'YearMonth', TARGET, 
    'satisfied_ticket', 'neutral_ticket', 'dissatisfied_ticket'
]
feature_cols = [c for c in merged.columns if c not in DROP_COLS]

X = merged[feature_cols]
y = merged[TARGET]

# Drop rows with NaN
mask = X.notna().all(axis=1) & y.notna()
X, y = X[mask].reset_index(drop=True), y[mask].reset_index(drop=True)
print(f"\nRows after NaN drop: {len(X)}")

# Compute Pearson Correlation
print("\nCalculating Pearson Correlation Matrix...")
corr_matrix = X.corr(method='pearson')

# Plot and save correlation matrix
plt.figure(figsize=(24, 20))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Pearson Correlation Matrix")
plt.savefig(os.path.join(base_dir, 'pearson_correlation_matrix.png'), dpi=150, bbox_inches='tight')
plt.close()

# Identify features with high collinearity (>0.7)
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper_tri.columns if any(upper_tri[column].abs() > 0.7)]
print(f"Features to drop due to multicollinearity (|corr| > 0.7): {to_drop}")

# Drop identified variables
X = X.drop(columns=to_drop)
feature_cols = X.columns.tolist()
print(f"Features after dropping multicollinear terms ({len(feature_cols)}): {feature_cols}")


# ═══════════════════════════════════════════════════════════
# SECTION 4 — TRAIN / TEST SPLIT
# ═══════════════════════════════════════════════════════════
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")


# ═══════════════════════════════════════════════════════════
# SECTION 5 — HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    m = y_true != 0
    return np.mean(np.abs((y_true[m] - y_pred[m]) / y_true[m])) * 100

def evaluate_model(name, model):
    return {
        'Model':      name,
        'Train MAE':  mean_absolute_error(y_train, model.predict(X_train)),
        'Test MAE':   mean_absolute_error(y_test,  model.predict(X_test)),
        'Train MAPE': mape(y_train, model.predict(X_train)),
        'Test MAPE':  mape(y_test,  model.predict(X_test)),
        'Train R2':   r2_score(y_train, model.predict(X_train)),
        'Test R2':    r2_score(y_test,  model.predict(X_test)),
    }


# ═══════════════════════════════════════════════════════════
# SECTION 6 — GRIDSEARCHCV MODEL TRAINING
# ═══════════════════════════════════════════════════════════
results     = []
best_models = {}

# ── Decision Tree ──
print("\n[1/3] Tuning Decision Tree...")
dt_gs = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    {'max_depth': [3, 5, 7, 10, None], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4]},
    cv=5, scoring='neg_mean_absolute_error', n_jobs=-1
)
dt_gs.fit(X_train, y_train)
best_models['Decision Tree'] = dt_gs.best_estimator_
results.append(evaluate_model('Decision Tree', dt_gs.best_estimator_))
print(f"   Best params: {dt_gs.best_params_}")

# ── Random Forest ──
print("\n[2/3] Tuning Random Forest...")
rf_gs = GridSearchCV(
    RandomForestRegressor(random_state=42),
    {'n_estimators': [100, 200, 300], 'max_depth': [5, 10, None], 'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2]},
    cv=5, scoring='neg_mean_absolute_error', n_jobs=-1
)
rf_gs.fit(X_train, y_train)
best_models['Random Forest'] = rf_gs.best_estimator_
results.append(evaluate_model('Random Forest', rf_gs.best_estimator_))
print(f"   Best params: {rf_gs.best_params_}")

# ── XGBoost ──
print("\n[3/3] Tuning XGBoost...")
xgb_gs = GridSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    {'n_estimators': [100, 200, 300], 'max_depth': [3, 5, 7], 'learning_rate': [0.01, 0.05, 0.1], 'subsample': [0.7, 1.0]},
    cv=5, scoring='neg_mean_absolute_error', n_jobs=-1
)
xgb_gs.fit(X_train, y_train)
best_models['XGBoost'] = xgb_gs.best_estimator_
results.append(evaluate_model('XGBoost', xgb_gs.best_estimator_))
print(f"   Best params: {xgb_gs.best_params_}")


# ═══════════════════════════════════════════════════════════
# SECTION 7 — EVALUATION TABLE
# ═══════════════════════════════════════════════════════════
results_df = pd.DataFrame(results).set_index('Model')
print("\n" + "="*70)
print("MODEL EVALUATION RESULTS")
print("="*70)
print(results_df[['Train MAE', 'Test MAE', 'Train MAPE', 'Test MAPE', 'Train R2', 'Test R2']].round(4))

results_df.to_csv(os.path.join(base_dir, 'model_results.csv'))


# ═══════════════════════════════════════════════════════════
# SECTION 8 — BUILT-IN FEATURE IMPORTANCE
# ═══════════════════════════════════════════════════════════
colors = ['#4C72B0', '#55A868', '#C44E52']

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle('Built-in Feature Importance', fontsize=15, fontweight='bold', y=1.01)

for ax, (name, model), color in zip(axes, best_models.items(), colors):
    imps       = model.feature_importances_
    sorted_idx = np.argsort(imps)
    ax.barh([feature_cols[i] for i in sorted_idx], imps[sorted_idx],
            color=color, alpha=0.85, edgecolor='white')
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance Score')
    ax.tick_params(axis='y', labelsize=9)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(base_dir, 'feature_importance_builtin.png'), dpi=150, bbox_inches='tight')
plt.close()


# ═══════════════════════════════════════════════════════════
# SECTION 9 — SHAP VALUES
# ═══════════════════════════════════════════════════════════
X_test_r = X_test.reset_index(drop=True)
shap_data = {}

print("\nCalculating SHAP values...")
for name, model in best_models.items():
    print(f"  ...{name}")
    try:
        exp = shap.TreeExplainer(model)
        shap_data[name] = exp.shap_values(X_test_r)
    except Exception as e:
        print(f"Error computing SHAP values for {name}: {e}")

# ── SHAP mean |value| bar chart ──
if shap_data:
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    fig.suptitle('SHAP Mean |Value| — Feature Importance', fontsize=15, fontweight='bold', y=1.01)

    for ax, (name, shap_vals), color in zip(axes, shap_data.items(), colors):
        if len(shap_vals.shape) == 2:
            mean_shap  = np.abs(shap_vals).mean(axis=0)
            sorted_idx = np.argsort(mean_shap)
            ax.barh([feature_cols[i] for i in sorted_idx], mean_shap[sorted_idx],
                    color=color, alpha=0.85, edgecolor='white')
            ax.set_title(f'{name} (SHAP)', fontsize=13, fontweight='bold')
            ax.set_xlabel('Mean |SHAP value|')
            ax.tick_params(axis='y', labelsize=9)
            ax.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(base_dir, 'shap_mean_importance.png'), dpi=150, bbox_inches='tight')
    plt.close()

# ── SHAP beeswarm per model ──
for name, model in best_models.items():
    try:
        exp         = shap.Explainer(model, X_train)
        shap_expl   = exp(X_test_r)
        
        plt.figure(figsize=(11, 7))
        shap.plots.beeswarm(shap_expl, show=False, max_display=len(feature_cols))
        plt.title(f'SHAP Beeswarm — {name}', fontsize=13, fontweight='bold')
        plt.tight_layout()
        safe_name = name.lower().replace(' ', '_')
        plt.savefig(os.path.join(base_dir, f'shap_beeswarm_{safe_name}.png'), dpi=150, bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"Error plotting SHAP beeswarm for {name}: {e}")

# ── SHAP summary table ──
shap_importance = pd.DataFrame({'Feature': feature_cols})
for name, shap_vals in shap_data.items():
    if len(shap_vals.shape) == 2:
        shap_importance[f'SHAP_{name}'] = np.abs(shap_vals).mean(axis=0)

if 'SHAP_XGBoost' in shap_importance.columns:
    shap_importance = shap_importance.sort_values('SHAP_XGBoost', ascending=False).reset_index(drop=True)

print("\nSHAP Feature Importance Table:")
print(shap_importance.round(4))

shap_importance.to_csv(os.path.join(base_dir, 'shap_importance_table.csv'), index=False)
print(f"\nAll outputs saved to {base_dir}")


Loading feature tables...
  NPS: (627, 10) | TXN: (963, 8) | DELIV: (942, 9) | PLANO: (757, 9) | SHOP: (164, 10)

Final merged dataset: (479, 39)
Months: ['2025-10', '2025-11', '2025-12', '2026-01', '2026-02']

Rows after NaN drop: 343

Calculating Pearson Correlation Matrix...
Features to drop due to multicollinearity (|corr| > 0.7): ['num_product', 'order_delivered', 'order_pickup', 'order_onsite', 'speed_median_hours', 'late_rate', 'err_basic', 'err_display', 'err_posm', 'sizeshop']
Features after dropping multicollinear terms (23): ['num_order', 'pct_delivered', 'order_delivered_tickets', 'order_pickup_tickets', 'order_onsite_tickets', 'order_other_tickets', 'speed_mean_hours', 'delay_mean_hours', 'delay_max_hours', 'promise_gap_mean_hours', 'ontime_rate', 'planogram_score', 'err_electronic_price_board', 'err_operations', 'err_advisory_partition', 'tong_dien_tichm2', 'rong_ngang_m', 'dai_sau_m', 'kv_ban_hangm2', 'kho_m2', 'via_he_rongm', 'is_southern', 'tenure']

Train: 274 | Test:

Background dataset has 274 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=274 when initializing the masker.
Background dataset has 274 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=274 when initializing the masker.
Background dataset has 274 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=274 when initializing the masker.


Error plotting SHAP beeswarm for Random Forest: Additivity check failed in TreeExplainer! Please ensure the data matrix you passed to the explainer is the same shape that the model was trained on. If your data shape is correct then please report this on GitHub. This check failed because for one of the samples the sum of the SHAP values was 1.297834, while the model output was 1.375297. If this difference is acceptable you can set check_additivity=False to disable this check.

SHAP Feature Importance Table:
                       Feature  SHAP_Decision Tree  SHAP_Random Forest  \
0              delay_max_hours              0.8071              0.8935   
1      order_delivered_tickets              0.3080              0.1015   
2             speed_mean_hours              0.2162              0.1105   
3          order_other_tickets              0.0947              0.1403   
4              planogram_score              0.0887              0.0680   
5             delay_mean_hours              

After 5-fold CV

In [18]:
# %%
# pip install matplotlib xgboost shap seaborn scikit-learn
# Data Manipulation and Analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Data Split & Tuning
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_validate

# Machine Learning Models (Regression)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from xgboost import XGBRegressor

# Model Evaluation Metrics (Regression)
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# Feature Importance
import shap

# Optional: Ignore warnings for cleaner outputs
import warnings
warnings.filterwarnings('ignore')
import os


# %%
# ═══════════════════════════════════════════════════════════
# SECTION 1 — LOAD ALL FEATURE TABLES
# ═══════════════════════════════════════════════════════════
print("Loading feature tables...")

base_dir = r'c:\Users\Phuc\Downloads\Thesis v2\Dataset\features'

nps   = pd.read_csv(os.path.join(base_dir, 'nps_aggregated_features.csv'))
txn   = pd.read_csv(os.path.join(base_dir, 'transaction_features.csv'))
deliv = pd.read_excel(os.path.join(base_dir, 'delivery_features.xlsx'))
plano = pd.read_csv(os.path.join(base_dir, 'planogram_features.csv'))
shop  = pd.read_csv(os.path.join(base_dir, 'shop.csv'))
ms  = pd.read_excel(os.path.join(base_dir, 'ms_features.xlsx'))

# Convert opening date to datetime if not already
shop['khai_truong_date'] = pd.to_datetime(shop['khai_truong_date'], errors='coerce')

print(f"  NPS: {nps.shape} | TXN: {txn.shape} | DELIV: {deliv.shape} | MS: {ms.shape} | PLANO: {plano.shape} | SHOP: {shop.shape}")

# ═══════════════════════════════════════════════════════════
# SECTION 2 — MERGE & TENURE
# ═══════════════════════════════════════════════════════════
merged = pd.merge(txn, nps, on=['store_id', 'YearMonth'], how='inner')
merged = pd.merge(merged, deliv, on=['store_id', 'YearMonth'], how='left')
merged = pd.merge(merged, plano, on=['store_id', 'YearMonth'], how='left')
merged = pd.merge(merged, ms, on=['store_id', 'YearMonth'], how='left')

# Filter to Oct 2025 – Feb 2026
merged = merged[merged['YearMonth'].between('2025-10', '2026-02')].copy()

# Merge with shop_info
merged = pd.merge(merged, shop, on='store_id', how='left')

# ── Create tenure column ──
merged['_period'] = pd.to_datetime(merged['YearMonth'], format='%Y-%m')
merged['tenure'] = (
    (merged['_period'].dt.year  - merged['khai_truong_date'].dt.year)  * 12 +
    (merged['_period'].dt.month - merged['khai_truong_date'].dt.month)
)
merged.drop(columns=['_period', 'khai_truong_date'], inplace=True)

print(f"\nFinal merged dataset: {merged.shape}")
print(f"Months: {sorted(merged['YearMonth'].unique())}")


# ═══════════════════════════════════════════════════════════
# SECTION 3 — FEATURE / TARGET SPLIT & PEARSON CORRELATION
# ═══════════════════════════════════════════════════════════
TARGET    = 'nps_weighted_score'

DROP_COLS = [
    'store_id', 'YearMonth', TARGET,
    'satisfied_ticket', 'neutral_ticket', 'dissatisfied_ticket'
]
feature_cols = [c for c in merged.columns if c not in DROP_COLS]

X = merged[feature_cols]
y = merged[TARGET]

# Drop rows with NaN
mask = X.notna().all(axis=1) & y.notna()
X, y = X[mask].reset_index(drop=True), y[mask].reset_index(drop=True)
print(f"\nRows after NaN drop: {len(X)}")

# ── MAPE reliability check (Fix #6) ──
near_zero_count = (y < 0.5).sum()
if near_zero_count > 0:
    print(f"\n⚠  MAPE WARNING: {near_zero_count} rows have target < 0.5.")
    print("   MAPE is unreliable for near-zero targets (diverges toward ∞).")
    print("   Treat MAE/RMSE as primary metrics in Section 3.8 Limitations.")

# Compute Pearson Correlation
print("\nCalculating Pearson Correlation Matrix...")
corr_matrix = X.corr(method='pearson')

plt.figure(figsize=(24, 20))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Pearson Correlation Matrix")
plt.savefig(os.path.join(base_dir, 'pearson_correlation_matrix.png'), dpi=150, bbox_inches='tight')
plt.close()

# Identify features with high collinearity (>0.7)
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper_tri.columns if any(upper_tri[column].abs() > 0.7)]
print(f"Features to drop due to multicollinearity (|corr| > 0.7): {to_drop}")

X = X.drop(columns=to_drop)
feature_cols = X.columns.tolist()
print(f"Features after dropping multicollinear terms ({len(feature_cols)}): {feature_cols}")


# ═══════════════════════════════════════════════════════════
# SECTION 4 — TRAIN / TEST SPLIT
# ═══════════════════════════════════════════════════════════
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")


# ═══════════════════════════════════════════════════════════
# SECTION 5 — HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    m = y_true != 0
    return np.mean(np.abs((y_true[m] - y_pred[m]) / y_true[m])) * 100

def evaluate_model(name, model):
    train_pred = model.predict(X_train)
    test_pred  = model.predict(X_test)
    return {
        'Model':      name,
        'Train MAE':  mean_absolute_error(y_train, train_pred),
        'Test MAE':   mean_absolute_error(y_test,  test_pred),
        'Train RMSE': np.sqrt(mean_squared_error(y_train, train_pred)),  # Fix #5
        'Test RMSE':  np.sqrt(mean_squared_error(y_test,  test_pred)),   # Fix #5
        'Train MAPE': mape(y_train, train_pred),
        'Test MAPE':  mape(y_test,  test_pred),
        'Train R2':   r2_score(y_train, train_pred),
        'Test R2':    r2_score(y_test,  test_pred),
    }


# ═══════════════════════════════════════════════════════════
# SECTION 6 — GRIDSEARCHCV MODEL TRAINING
# ═══════════════════════════════════════════════════════════
results     = []
best_models = {}

# ── Naïve Baseline (Fix #4) ──
print("\n[0/3] Evaluating Naïve Baseline (mean predictor)...")
dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train, y_train)
best_models['Baseline (Mean)'] = dummy
results.append(evaluate_model('Baseline (Mean)', dummy))

# ── Decision Tree (Fix #1: cap depth, raise min_samples_leaf) ──
print("\n[1/3] Tuning Decision Tree...")
dt_gs = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    {
        'max_depth':         [2, 3, 4],       # removed 5, 7, 10, None — prevents memorisation
        'min_samples_split': [5, 10],
        'min_samples_leaf':  [5, 10, 15],     # raised floor from 1→5
    },
    cv=5, scoring='neg_mean_absolute_error', n_jobs=-1
)
dt_gs.fit(X_train, y_train)
best_models['Decision Tree'] = dt_gs.best_estimator_
results.append(evaluate_model('Decision Tree', dt_gs.best_estimator_))
print(f"   Best params: {dt_gs.best_params_}")

# ── Random Forest ──
print("\n[2/3] Tuning Random Forest...")
rf_gs = GridSearchCV(
    RandomForestRegressor(random_state=42),
    {
        'n_estimators':    [100, 200, 300],
        'max_depth':       [5, 10, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf':  [1, 2],
    },
    cv=5, scoring='neg_mean_absolute_error', n_jobs=-1
)
rf_gs.fit(X_train, y_train)
best_models['Random Forest'] = rf_gs.best_estimator_
results.append(evaluate_model('Random Forest', rf_gs.best_estimator_))
print(f"   Best params: {rf_gs.best_params_}")

# ── XGBoost (Fix #2: add regularisation to base estimator) ──
print("\n[3/3] Tuning XGBoost...")
xgb_gs = GridSearchCV(
    XGBRegressor(
        random_state=42, verbosity=0,
        min_child_weight=5,   # prevents fitting on tiny leaf nodes
        reg_lambda=1.5,        # L2 regularisation
        reg_alpha=0.5,         # L1 regularisation
    ),
    {
        'n_estimators':  [100, 200, 300],
        'max_depth':     [2, 3],           # constrained — was [3,5,7]
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample':     [0.7, 0.8],
    },
    cv=5, scoring='neg_mean_absolute_error', n_jobs=-1
)
xgb_gs.fit(X_train, y_train)
best_models['XGBoost'] = xgb_gs.best_estimator_
results.append(evaluate_model('XGBoost', xgb_gs.best_estimator_))
print(f"   Best params: {xgb_gs.best_params_}")


# ═══════════════════════════════════════════════════════════
# SECTION 7 — EVALUATION TABLE (train/test split)
# ═══════════════════════════════════════════════════════════
results_df = pd.DataFrame(results).set_index('Model')
print("\n" + "="*70)
print("MODEL EVALUATION RESULTS (Train/Test Split)")
print("="*70)
print(results_df[[
    'Train MAE', 'Test MAE',
    'Train RMSE', 'Test RMSE',
    'Train MAPE', 'Test MAPE',
    'Train R2', 'Test R2'
]].round(4))

results_df.to_csv(os.path.join(base_dir, 'model_results.csv'))


# ═══════════════════════════════════════════════════════════
# SECTION 7B — 5-FOLD CROSS-VALIDATION
# ═══════════════════════════════════════════════════════════
print("\n" + "="*70)
print("5-FOLD CROSS-VALIDATION RESULTS (Full Dataset, mean ± std)")
print("="*70)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_summary = []

for name, model in best_models.items():
    cv_res = cross_validate(
        model, X, y, cv=kf,
        scoring=['r2', 'neg_mean_absolute_error', 'neg_mean_squared_error'],
        return_train_score=True   # ← thêm dòng này
    )
    rmse_test_per_fold  = np.sqrt(-cv_res['test_neg_mean_squared_error'])
    rmse_train_per_fold = np.sqrt(-cv_res['train_neg_mean_squared_error'])

    cv_summary.append({
        'Model':          name,
        # ── Test metrics ──
        'Test R² mean':   cv_res['test_r2'].mean(),
        'Test R² std':    cv_res['test_r2'].std(),
        'Test MAE mean':  -cv_res['test_neg_mean_absolute_error'].mean(),
        'Test MAE std':    cv_res['test_neg_mean_absolute_error'].std(),
        'Test RMSE mean': rmse_test_per_fold.mean(),
        'Test RMSE std':  rmse_test_per_fold.std(),
        # ── Train metrics ──
        'Train R² mean':   cv_res['train_r2'].mean(),
        'Train R² std':    cv_res['train_r2'].std(),
        'Train MAE mean':  -cv_res['train_neg_mean_absolute_error'].mean(),
        'Train MAE std':    cv_res['train_neg_mean_absolute_error'].std(),
        'Train RMSE mean': rmse_train_per_fold.mean(),
        'Train RMSE std':  rmse_train_per_fold.std(),
    })

cv_df = pd.DataFrame(cv_summary).set_index('Model')
print(cv_df.round(4))

print("\nThesis-ready format (mean ± std over 5 folds):")
for idx, row in cv_df.iterrows():
    print(f"\n  {idx}")
    print(
        f"    Train  →  "
        f"R² = {row['Train R² mean']:.4f} ± {row['Train R² std']:.4f}  |  "
        f"MAE = {row['Train MAE mean']:.4f} ± {row['Train MAE std']:.4f}  |  "
        f"RMSE = {row['Train RMSE mean']:.4f} ± {row['Train RMSE std']:.4f}"
    )
    print(
        f"    Test   →  "
        f"R² = {row['Test R² mean']:.4f} ± {row['Test R² std']:.4f}  |  "
        f"MAE = {row['Test MAE mean']:.4f} ± {row['Test MAE std']:.4f}  |  "
        f"RMSE = {row['Test RMSE mean']:.4f} ± {row['Test RMSE std']:.4f}"
    )

cv_df.to_csv(os.path.join(base_dir, 'model_cv_results.csv'))
print(f"\nCV results saved to model_cv_results.csv")


# ═══════════════════════════════════════════════════════════
# SECTION 8 — BUILT-IN FEATURE IMPORTANCE
# (skip Baseline — it has no feature_importances_)
# ═══════════════════════════════════════════════════════════
tree_models = {k: v for k, v in best_models.items() if k != 'Baseline (Mean)'}
colors = ['#4C72B0', '#55A868', '#C44E52']

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle('Built-in Feature Importance', fontsize=15, fontweight='bold', y=1.01)

for ax, (name, model), color in zip(axes, tree_models.items(), colors):
    imps       = model.feature_importances_
    sorted_idx = np.argsort(imps)
    ax.barh([feature_cols[i] for i in sorted_idx], imps[sorted_idx],
            color=color, alpha=0.85, edgecolor='white')
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance Score')
    ax.tick_params(axis='y', labelsize=9)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(base_dir, 'feature_importance_builtin.png'), dpi=150, bbox_inches='tight')
plt.close()


# ═══════════════════════════════════════════════════════════
# SECTION 9 — SHAP VALUES
# ═══════════════════════════════════════════════════════════
X_test_r = X_test.reset_index(drop=True)
shap_data = {}

print("\nCalculating SHAP values...")
for name, model in tree_models.items():
    print(f"  ...{name}")
    try:
        exp = shap.TreeExplainer(model)
        shap_data[name] = exp.shap_values(X_test_r)
    except Exception as e:
        print(f"Error computing SHAP values for {name}: {e}")

# ── SHAP mean |value| bar chart ──
if shap_data:
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    fig.suptitle('SHAP Mean |Value| — Feature Importance', fontsize=15, fontweight='bold', y=1.01)

    for ax, (name, shap_vals), color in zip(axes, shap_data.items(), colors):
        if len(shap_vals.shape) == 2:
            mean_shap  = np.abs(shap_vals).mean(axis=0)
            sorted_idx = np.argsort(mean_shap)
            ax.barh([feature_cols[i] for i in sorted_idx], mean_shap[sorted_idx],
                    color=color, alpha=0.85, edgecolor='white')
            ax.set_title(f'{name} (SHAP)', fontsize=13, fontweight='bold')
            ax.set_xlabel('Mean |SHAP value|')
            ax.tick_params(axis='y', labelsize=9)
            ax.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(base_dir, 'shap_mean_importance.png'), dpi=150, bbox_inches='tight')
    plt.close()

# ── SHAP beeswarm per model ──
for name, model in tree_models.items():
    try:
        exp       = shap.Explainer(model, X_train)
        shap_expl = exp(X_test_r)

        plt.figure(figsize=(11, 7))
        shap.plots.beeswarm(shap_expl, show=False, max_display=len(feature_cols))
        plt.title(f'SHAP Beeswarm — {name}', fontsize=13, fontweight='bold')
        plt.tight_layout()
        safe_name = name.lower().replace(' ', '_')
        plt.savefig(os.path.join(base_dir, f'shap_beeswarm_{safe_name}.png'), dpi=150, bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"Error plotting SHAP beeswarm for {name}: {e}")

# ── SHAP summary table ──
shap_importance = pd.DataFrame({'Feature': feature_cols})
for name, shap_vals in shap_data.items():
    if len(shap_vals.shape) == 2:
        shap_importance[f'SHAP_{name}'] = np.abs(shap_vals).mean(axis=0)

if 'SHAP_XGBoost' in shap_importance.columns:
    shap_importance = shap_importance.sort_values('SHAP_XGBoost', ascending=False).reset_index(drop=True)

print("\nSHAP Feature Importance Table:")
print(shap_importance.round(4))

shap_importance.to_csv(os.path.join(base_dir, 'shap_importance_table.csv'), index=False)
print(f"\nAll outputs saved to {base_dir}")


Loading feature tables...
  NPS: (627, 10) | TXN: (963, 8) | DELIV: (942, 9) | MS: (938, 7) | PLANO: (757, 9) | SHOP: (164, 10)

Final merged dataset: (479, 44)
Months: ['2025-10', '2025-11', '2025-12', '2026-01', '2026-02']

Rows after NaN drop: 343

Calculating Pearson Correlation Matrix...
Features to drop due to multicollinearity (|corr| > 0.7): ['num_product', 'order_delivered', 'order_pickup', 'order_onsite', 'speed_median_hours', 'late_rate', 'err_basic', 'err_display', 'err_posm', 'sizeshop']
Features after dropping multicollinear terms (28): ['num_order', 'pct_delivered', 'order_delivered_tickets', 'order_pickup_tickets', 'order_onsite_tickets', 'order_other_tickets', 'speed_mean_hours', 'delay_mean_hours', 'delay_max_hours', 'promise_gap_mean_hours', 'ontime_rate', 'planogram_score', 'err_electronic_price_board', 'err_operations', 'err_advisory_partition', 'PA_score', 'R_score', 'PI_score', 'PS_score', 'PO_score', 'tong_dien_tichm2', 'rong_ngang_m', 'dai_sau_m', 'kv_ban_hangm

Background dataset has 274 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=274 when initializing the masker.
Background dataset has 274 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=274 when initializing the masker.
Background dataset has 274 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=274 when initializing the masker.


Error plotting SHAP beeswarm for Random Forest: Additivity check failed in TreeExplainer! Please ensure the data matrix you passed to the explainer is the same shape that the model was trained on. If your data shape is correct then please report this on GitHub. This check failed because for one of the samples the sum of the SHAP values was 1.230715, while the model output was 1.347633. If this difference is acceptable you can set check_additivity=False to disable this check.

SHAP Feature Importance Table:
                       Feature  SHAP_Decision Tree  SHAP_Random Forest  \
0              delay_max_hours              1.0388              0.9023   
1          order_other_tickets              0.1618              0.1155   
2             speed_mean_hours              0.0141              0.1163   
3       promise_gap_mean_hours              0.0892              0.0154   
4              planogram_score              0.0000              0.0453   
5      order_delivered_tickets              

In [17]:
# %%
# pip install matplotlib xgboost shap seaborn scikit-learn
# Data Manipulation and Analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Data Split & Tuning
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV

# Machine Learning Models (Regression)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Model Evaluation Metrics (Regression)
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# Feature Importance
import shap

# Optional: Ignore warnings for cleaner outputs
import warnings
warnings.filterwarnings('ignore')
import os


# %%
# ═══════════════════════════════════════════════════════════
# SECTION 1 — LOAD ALL FEATURE TABLES
# ═══════════════════════════════════════════════════════════
print("Loading feature tables...")

base_dir = r'c:\Users\Phuc\Downloads\Thesis v2\Dataset\features'
out_dir = r'c:\Users\Phuc\Downloads\Thesis v2\Dataset\features_random'
os.makedirs(out_dir, exist_ok=True)

nps   = pd.read_csv(os.path.join(base_dir, 'nps_aggregated_features.csv'))
txn   = pd.read_csv(os.path.join(base_dir, 'transaction_features.csv'))
deliv = pd.read_excel(os.path.join(base_dir, 'delivery_features.xlsx'))
plano = pd.read_csv(os.path.join(base_dir, 'planogram_features.csv'))
shop  = pd.read_csv(os.path.join(base_dir, 'shop.csv')) 
ms = pd.read_excel(os.path.join(base_dir, 'ms_features.xlsx'))

# Convert opening date to datetime if not already
shop['khai_truong_date'] = pd.to_datetime(shop['khai_truong_date'], errors='coerce')


print(f"  NPS: {nps.shape} | TXN: {txn.shape} | DELIV: {deliv.shape} | PLANO: {plano.shape} | SHOP: {shop.shape}")

# ═══════════════════════════════════════════════════════════
# SECTION 2 — MERGE & TENURE
# ═══════════════════════════════════════════════════════════
# Merge datasets starting from a base combination (txn and nps)
merged = pd.merge(txn, nps, on=['store_id', 'YearMonth'], how='inner')
merged = pd.merge(merged, deliv, on=['store_id', 'YearMonth'], how='left')  
merged = pd.merge(merged, plano, on=['store_id', 'YearMonth'], how='left')

# Filter to Oct 2025 – Feb 2026
merged = merged[merged['YearMonth'].between('2025-10', '2026-02')].copy()

# Merge with shop_info
merged = pd.merge(merged, shop, on='store_id', how='left')

# ── Create tenure column ──
merged['_period'] = pd.to_datetime(merged['YearMonth'], format='%Y-%m')
merged['tenure'] = (
    (merged['_period'].dt.year  - merged['khai_truong_date'].dt.year)  * 12 +
    (merged['_period'].dt.month - merged['khai_truong_date'].dt.month)
)
merged.drop(columns=['_period', 'khai_truong_date'], inplace=True)

print(f"\nFinal merged dataset: {merged.shape}")
print(f"Months: {sorted(merged['YearMonth'].unique())}")


# ═══════════════════════════════════════════════════════════
# SECTION 3 — FEATURE / TARGET SPLIT & PEARSON CORRELATION
# ═══════════════════════════════════════════════════════════
TARGET    = 'nps_weighted_score'

# Exclude identification columns and calculation intermediates from features
DROP_COLS = [
    'store_id', 'YearMonth', TARGET, 
    'satisfied_ticket', 'neutral_ticket', 'dissatisfied_ticket'
]
feature_cols = [c for c in merged.columns if c not in DROP_COLS]

X = merged[feature_cols]
y = merged[TARGET]

# Drop rows with NaN
mask = X.notna().all(axis=1) & y.notna()
X, y = X[mask].reset_index(drop=True), y[mask].reset_index(drop=True)
print(f"\nRows after NaN drop: {len(X)}")

# Compute Pearson Correlation
print("\nCalculating Pearson Correlation Matrix...")
corr_matrix = X.corr(method='pearson')

# Plot and save correlation matrix
plt.figure(figsize=(24, 20))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Pearson Correlation Matrix")
plt.savefig(os.path.join(out_dir, 'pearson_correlation_matrix.png'), dpi=150, bbox_inches='tight')
plt.close()

# Identify features with high collinearity (>0.7)
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper_tri.columns if any(upper_tri[column].abs() > 0.7)]
print(f"Features to drop due to multicollinearity (|corr| > 0.7): {to_drop}")

# Drop identified variables
X = X.drop(columns=to_drop)
feature_cols = X.columns.tolist()
print(f"Features after dropping multicollinear terms ({len(feature_cols)}): {feature_cols}")


# ═══════════════════════════════════════════════════════════
# SECTION 4 — TRAIN / TEST SPLIT
# ═══════════════════════════════════════════════════════════
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")


# ═══════════════════════════════════════════════════════════
# SECTION 5 — HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    m = y_true != 0
    return np.mean(np.abs((y_true[m] - y_pred[m]) / y_true[m])) * 100

def evaluate_model(name, model):
    return {
        'Model':      name,
        'Train MAE':  mean_absolute_error(y_train, model.predict(X_train)),
        'Test MAE':   mean_absolute_error(y_test,  model.predict(X_test)),
        'Train MAPE': mape(y_train, model.predict(X_train)),
        'Test MAPE':  mape(y_test,  model.predict(X_test)),
        'Train R2':   r2_score(y_train, model.predict(X_train)),
        'Test R2':    r2_score(y_test,  model.predict(X_test)),
    }


# ═══════════════════════════════════════════════════════════
# SECTION 6 — RANDOMIZEDSEARCHCV MODEL TRAINING
# ═══════════════════════════════════════════════════════════
results     = []
best_models = {}

# ── Decision Tree ──
print("\n[1/3] Tuning Decision Tree...")
dt_gs = RandomizedSearchCV(
    DecisionTreeRegressor(random_state=42),
    {
        'max_depth':         [3, 5, 7, None],   # bỏ 10: với N=427, depth=10 ≈ None, thừa
        'min_samples_split': [5, 10, 20],        # tăng lower bound từ 2→5: split trên 2 obs quá nhỏ
        'min_samples_leaf':  [2, 4, 6],          # tăng lower bound từ 1→2: leaf=1 = memorize
        'ccp_alpha':         [0.0, 0.01, 0.05], # thêm cost-complexity pruning: regularize sau khi grow
    },
    n_iter=30,          # tăng từ 10→30: param space lớn hơn, cần sample nhiều hơn
    cv=5,
    random_state=42,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)
dt_gs.fit(X_train, y_train)
best_models['Decision Tree'] = dt_gs.best_estimator_
results.append(evaluate_model('Decision Tree', dt_gs.best_estimator_))
print(f"   Best params: {dt_gs.best_params_}")

# ── Random Forest ──
print("\n[2/3] Tuning Random Forest...")
rf_gs = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    {
        'n_estimators':      [200, 300, 500],    # bỏ 100: RF cần đủ trees để variance reduction ổn định
        'max_depth':         [5, 10, 15, None],  # thêm 15: cho phép search rộng hơn giữa shallow và unlimited
        'min_samples_split': [5, 10],            # tăng lower bound từ 2→5: lý do như DT
        'min_samples_leaf':  [2, 4],             # tăng lower bound từ 1→2: lý do như DT
        'max_features':      ['sqrt', 0.5, 0.7], # thêm param: RF dùng random feature subset, đây là lever regularization chính
    },
    n_iter=30,
    cv=5,
    random_state=42,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)
rf_gs.fit(X_train, y_train)
best_models['Random Forest'] = rf_gs.best_estimator_
results.append(evaluate_model('Random Forest', rf_gs.best_estimator_))
print(f"   Best params: {rf_gs.best_params_}")

# ── XGBoost ──
print("\n[3/3] Tuning XGBoost...")
xgb_gs = RandomizedSearchCV(
    XGBRegressor(
        random_state=42,
        verbosity=0,
        # Fixed regularization — KHÔNG đưa vào search space
        # Lý do: với N=427, những params này cần được constrain cứng
        # thay vì để CV tự chọn, vì CV trên small dataset vẫn có thể
        # chọn giá trị quá permissive nếu một fold tình cờ "may mắn"
        min_child_weight=5,    # node cần ≥5 obs để split: ngăn overfit trên leaf nhỏ
        reg_lambda=1.5,        # L2: shrink weights, giảm sensitivity với individual obs
        reg_alpha=0.5,         # L1: sparse feature weights, ít features "active" hơn
        colsample_bytree=0.8,  # dùng 80% features mỗi tree: tương đương max_features của RF
    ),
    param_distributions={
        'n_estimators':  [100, 200, 300],
        'max_depth':     [2, 3],           # HARD CAP ở 3: boosting với shallow trees hiệu quả hơn deep trees
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample':     [0.7, 0.8],       # bỏ 1.0: subsample=1.0 = không có row sampling = dễ overfit
    },
    n_iter=30,
    cv=5,
    random_state=42,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)
xgb_gs.fit(X_train, y_train)
best_models['XGBoost'] = xgb_gs.best_estimator_
results.append(evaluate_model('XGBoost', xgb_gs.best_estimator_))
print(f"   Best params: {xgb_gs.best_params_}")


# ═══════════════════════════════════════════════════════════
# SECTION 7 — EVALUATION TABLE
# ═══════════════════════════════════════════════════════════
results_df = pd.DataFrame(results).set_index('Model')
print("\n" + "="*70)
print("MODEL EVALUATION RESULTS")
print("="*70)
print(results_df[['Train MAE', 'Test MAE', 'Train MAPE', 'Test MAPE', 'Train R2', 'Test R2']].round(4))

results_df.to_csv(os.path.join(out_dir, 'model_results.csv'))


# ═══════════════════════════════════════════════════════════
# SECTION 8 — BUILT-IN FEATURE IMPORTANCE
# ═══════════════════════════════════════════════════════════
colors = ['#4C72B0', '#55A868', '#C44E52']

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle('Built-in Feature Importance', fontsize=15, fontweight='bold', y=1.01)

for ax, (name, model), color in zip(axes, best_models.items(), colors):
    imps       = model.feature_importances_
    sorted_idx = np.argsort(imps)
    ax.barh([feature_cols[i] for i in sorted_idx], imps[sorted_idx],
            color=color, alpha=0.85, edgecolor='white')
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance Score')
    ax.tick_params(axis='y', labelsize=9)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'feature_importance_builtin.png'), dpi=150, bbox_inches='tight')
plt.close()


# ═══════════════════════════════════════════════════════════
# SECTION 9 — SHAP VALUES
# ═══════════════════════════════════════════════════════════
X_test_r = X_test.reset_index(drop=True)
shap_data = {}

print("\nCalculating SHAP values...")
for name, model in best_models.items():
    print(f"  ...{name}")
    try:
        exp = shap.TreeExplainer(model)
        shap_data[name] = exp.shap_values(X_test_r)
    except Exception as e:
        print(f"Error computing SHAP values for {name}: {e}")

# ── SHAP mean |value| bar chart ──
if shap_data:
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    fig.suptitle('SHAP Mean |Value| — Feature Importance', fontsize=15, fontweight='bold', y=1.01)

    for ax, (name, shap_vals), color in zip(axes, shap_data.items(), colors):
        if len(shap_vals.shape) == 2:
            mean_shap  = np.abs(shap_vals).mean(axis=0)
            sorted_idx = np.argsort(mean_shap)
            ax.barh([feature_cols[i] for i in sorted_idx], mean_shap[sorted_idx],
                    color=color, alpha=0.85, edgecolor='white')
            ax.set_title(f'{name} (SHAP)', fontsize=13, fontweight='bold')
            ax.set_xlabel('Mean |SHAP value|')
            ax.tick_params(axis='y', labelsize=9)
            ax.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'shap_mean_importance.png'), dpi=150, bbox_inches='tight')
    plt.close()

# ── SHAP beeswarm per model ──
for name, model in best_models.items():
    try:
        exp         = shap.Explainer(model, X_train)
        shap_expl   = exp(X_test_r)
        
        plt.figure(figsize=(11, 7))
        shap.plots.beeswarm(shap_expl, show=False, max_display=len(feature_cols))
        plt.title(f'SHAP Beeswarm — {name}', fontsize=13, fontweight='bold')
        plt.tight_layout()
        safe_name = name.lower().replace(' ', '_')
        plt.savefig(os.path.join(out_dir, f'shap_beeswarm_{safe_name}.png'), dpi=150, bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"Error plotting SHAP beeswarm for {name}: {e}")

# ── SHAP summary table ──
shap_importance = pd.DataFrame({'Feature': feature_cols})
for name, shap_vals in shap_data.items():
    if len(shap_vals.shape) == 2:
        shap_importance[f'SHAP_{name}'] = np.abs(shap_vals).mean(axis=0)

if 'SHAP_XGBoost' in shap_importance.columns:
    shap_importance = shap_importance.sort_values('SHAP_XGBoost', ascending=False).reset_index(drop=True)

print("\nSHAP Feature Importance Table:")
print(shap_importance.round(4))

shap_importance.to_csv(os.path.join(out_dir, 'shap_importance_table.csv'), index=False)
print(f"\nAll outputs saved to {base_dir}")


Loading feature tables...
  NPS: (627, 10) | TXN: (963, 8) | DELIV: (942, 9) | PLANO: (757, 9) | SHOP: (164, 10)

Final merged dataset: (479, 39)
Months: ['2025-10', '2025-11', '2025-12', '2026-01', '2026-02']

Rows after NaN drop: 343

Calculating Pearson Correlation Matrix...
Features to drop due to multicollinearity (|corr| > 0.7): ['num_product', 'order_delivered', 'order_pickup', 'order_onsite', 'speed_median_hours', 'late_rate', 'err_basic', 'err_display', 'err_posm', 'sizeshop']
Features after dropping multicollinear terms (23): ['num_order', 'pct_delivered', 'order_delivered_tickets', 'order_pickup_tickets', 'order_onsite_tickets', 'order_other_tickets', 'speed_mean_hours', 'delay_mean_hours', 'delay_max_hours', 'promise_gap_mean_hours', 'ontime_rate', 'planogram_score', 'err_electronic_price_board', 'err_operations', 'err_advisory_partition', 'tong_dien_tichm2', 'rong_ngang_m', 'dai_sau_m', 'kv_ban_hangm2', 'kho_m2', 'via_he_rongm', 'is_southern', 'tenure']

Train: 274 | Test:

Background dataset has 274 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=274 when initializing the masker.
Background dataset has 274 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=274 when initializing the masker.
Background dataset has 274 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=274 when initializing the masker.



SHAP Feature Importance Table:
                       Feature  SHAP_Decision Tree  SHAP_Random Forest  \
0              delay_max_hours              0.9816              0.7528   
1          order_other_tickets              0.1638              0.2908   
2      order_delivered_tickets              0.0000              0.0989   
3             delay_mean_hours              0.0000              0.0420   
4              planogram_score              0.1493              0.0517   
5             speed_mean_hours              0.0148              0.0402   
6       promise_gap_mean_hours              0.0898              0.0395   
7                 via_he_rongm              0.0000              0.0041   
8                kv_ban_hangm2              0.0929              0.0133   
9                    num_order              0.0012              0.0106   
10  err_electronic_price_board              0.0000              0.0125   
11                      tenure              0.0386              0.0121   
12    